# Hills · Running Water · Concrete · Wood · Pervious Neighborhood · Light Pipe

> **From terrain to structure to optics** — one coherent physical environment.

| § | Domain | Core Physics |
|---|---|---|
| §1 | 🏔️ Hills | Perlin terrain, hill shading, slope/aspect, drainage basins |
| §2 | 🌊 Running Water | Shallow water eqns, Manning's n, river routing, discharge |
| §3 | 🏗️ Concrete | Stress-strain, compressive strength, reinforced beam, creep |
| §4 | 🪵 Wood | Orthotropic elasticity, grain texture, Weibull failure, moisture |
| §5 | 🏘️ Pervious Neighborhood | SCS curve number, Green-Ampt infiltration, stormwater routing |
| §6 | 💡 Light Pipe | TIR, NA, acceptance cone, luminance transport, etendue |


In [ ]:
import sympy as sp
import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import LightSource, Normalize
from mpl_toolkits.mplot3d import Axes3D
from scipy.ndimage import gaussian_filter, label
from scipy.integrate import solve_ivp
from scipy.stats import weibull_min
from IPython.display import display, Math
import warnings; warnings.filterwarnings('ignore')
sp.init_printing(use_latex='mathjax')
plt.rcParams.update({'font.size':10,'figure.dpi':100})
torch.set_default_dtype(torch.float64)
np.random.seed(2026)
print('ready')

---
# §1 🏔️ Hills — Terrain Generation, Hill Shading, Drainage Basins

**Multi-octave Perlin noise** → elevation $z(x,y)$.  
**Hill shading** (analytical): $I = \\hat{n}\\cdot\\hat{\\ell}$ where
$\\hat{n} = (-\\partial z/\\partial x,\\,-\\partial z/\\partial y,\\,1)/|\\cdots|$.  
**Slope & aspect:** $S=\\arctan\\sqrt{(\\partial z/\\partial x)^2+(\\partial z/\\partial y)^2}$,
$A=\\arctan2(-\\partial z/\\partial y,\\,\\partial z/\\partial x)$.  
**Drainage basins:** flow accumulation via D8 algorithm — each cell drains to
the steepest downslope neighbor.

In [ ]:
# ── Multi-octave Perlin noise ─────────────────────────────────────
def fade(t):     return 6*t**5 - 15*t**4 + 10*t**3
def lerp(a,b,t): return a + t*(b-a)

def perlin2d_oct(shape, octaves, persistence=0.5, lacunarity=2.0, seed=0):
    """Multi-octave Perlin noise returning elevation array."""
    H, W = shape; terrain = np.zeros(H*W)
    freq=1.0; amp=1.0; total_amp=0.0
    for oct_i in range(octaves):                # loop: add octaves
        rng = np.random.default_rng(seed+oct_i)
        angles = rng.uniform(0,2*np.pi,256)
        grads  = np.stack([np.cos(angles),np.sin(angles)],1)
        perm   = rng.permutation(256)
        # Grid
        gx,gy = np.meshgrid(np.linspace(0,freq*4,W),np.linspace(0,freq*4,H))
        xf=gx.ravel(); yf=gy.ravel()
        xi=xf.astype(int)&255; yi=yf.astype(int)&255
        xd=xf-xf.astype(int); yd=yf-yf.astype(int)
        u=fade(xd); v=fade(yd)
        def g2(ix,iy,dx,dy):
            g=grads[(perm[ix&255]+iy)&255]
            return g[:,0]*dx+g[:,1]*dy
        n00=g2(xi,yi,xd,yd); n10=g2(xi+1,yi,xd-1,yd)
        n01=g2(xi,yi+1,xd,yd-1); n11=g2(xi+1,yi+1,xd-1,yd-1)
        layer = lerp(lerp(n00,n10,u),lerp(n01,n11,u),v)
        terrain += amp*layer; total_amp+=amp
        freq*=lacunarity; amp*=persistence
    return (terrain/total_amp).reshape(H,W)

# Build 512×512 terrain
N_terrain = 512
elev = perlin2d_oct((N_terrain,N_terrain), octaves=7, persistence=0.55, seed=42)
# Scale to realistic elevation (0–800 m)
elev = (elev - elev.min())/(elev.max()-elev.min()) * 800.0

# ── Gradient (slope & aspect) ──────────────────────────────────────
dx_m = 10.0   # 10 m/pixel
dzdx = np.gradient(elev, dx_m, axis=1)
dzdy = np.gradient(elev, dx_m, axis=0)
slope_rad  = np.arctan(np.sqrt(dzdx**2 + dzdy**2))
slope_deg  = np.degrees(slope_rad)
aspect_deg = np.degrees(np.arctan2(-dzdy, dzdx)) % 360

# ── Hill shading ─────────────────────────────────────────────────
ls = LightSource(azdeg=315, altdeg=45)   # NW sun
hillshade = ls.hillshade(elev, vert_exag=2.0, dx=dx_m, dy=dx_m)

# ── D8 flow accumulation (drainage) ──────────────────────────────
# Downsample for speed
DSAMP = 4
elev_ds = elev[::DSAMP, ::DSAMP].copy()
Hd, Wd  = elev_ds.shape

# D8 directions: 8 neighbors
dirs_d8 = [(-1,-1),(-1,0),(-1,1),(0,-1),(0,1),(1,-1),(1,0),(1,1)]
dist_d8 = [np.sqrt(2),1,np.sqrt(2),1,1,np.sqrt(2),1,np.sqrt(2)]

flow_dir = np.full((Hd,Wd), -1, dtype=int)
for r in range(1,Hd-1):            # loop: compute flow direction
    for c in range(1,Wd-1):        # loop: per cell
        max_drop=-1e9; best=-1
        for d_idx,(dr,dc) in enumerate(dirs_d8):   # loop: 8 neighbors
            nr,nc=r+dr,c+dc
            drop=(elev_ds[r,c]-elev_ds[nr,nc])/dist_d8[d_idx]
            if drop>max_drop: max_drop=drop; best=d_idx
        flow_dir[r,c]=best

# Flow accumulation via topological sort
accum = np.ones((Hd,Wd))
# Sort cells by elevation (high → low)
order = np.argsort(elev_ds.ravel())[::-1]
for flat_idx in order:              # loop: pour water downhill
    r,c = flat_idx//Wd, flat_idx%Wd
    if flow_dir[r,c]>=0:
        dr,dc=dirs_d8[flow_dir[r,c]]
        nr,nc=r+dr,c+dc
        if 0<=nr<Hd and 0<=nc<Wd:
            accum[nr,nc]+=accum[r,c]

# Rivers where accumulation > threshold
river_mask = accum > 200

# ── Plot ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(2,3,figsize=(15,9))

im0=axes[0][0].imshow(elev,cmap='terrain',origin='lower')
axes[0][0].set_title('Elevation (m)'); plt.colorbar(im0,ax=axes[0][0],shrink=0.8)

axes[0][1].imshow(hillshade,cmap='gray',origin='lower')
axes[0][1].imshow(elev,cmap='terrain',origin='lower',alpha=0.45)
axes[0][1].set_title('Hillshade + terrain')

im2=axes[0][2].imshow(slope_deg,cmap='YlOrRd',origin='lower')
axes[0][2].set_title('Slope (degrees)'); plt.colorbar(im2,ax=axes[0][2],shrink=0.8)

im3=axes[1][0].imshow(aspect_deg,cmap='hsv',origin='lower')
axes[1][0].set_title('Aspect (degrees)'); plt.colorbar(im3,ax=axes[1][0],shrink=0.8)

axes[1][1].imshow(np.log1p(accum),cmap='Blues',origin='lower')
axes[1][1].imshow(river_mask,cmap='cool',origin='lower',alpha=0.6)
axes[1][1].set_title('Flow accumulation + rivers')

# 3D surface (subsampled)
S=8; xs_3d=np.arange(0,N_terrain,S); ys_3d=np.arange(0,N_terrain,S)
Xs,Ys=np.meshgrid(xs_3d,ys_3d)
Zs=elev[::S,::S]
ax3=axes[1][2]; ax3.remove(); ax3=fig.add_subplot(2,3,6,projection='3d')
ax3.plot_surface(Xs,Ys,Zs,cmap='terrain',rstride=1,cstride=1,
                  linewidth=0,antialiased=False,alpha=0.9)
ax3.set_title('3D terrain')
ax3.set_xlabel('x'); ax3.set_ylabel('y'); ax3.set_zlabel('z (m)')

plt.suptitle('🏔️ Hills: Perlin Terrain + Hill Shading + Drainage',
             fontsize=12,fontweight='bold')
plt.tight_layout(); plt.show()
print(f'Terrain: {N_terrain}×{N_terrain}, elev range {elev.min():.0f}–{elev.max():.0f} m')
print(f'Mean slope: {slope_deg.mean():.1f}°,  River pixels: {river_mask.sum():,}')

---
# §2 🌊 Running Water — Shallow Water Equations, Manning, Discharge

**Shallow water equations (1D):**
$$\\frac{\\partial h}{\\partial t}+\\frac{\\partial(hu)}{\\partial x}=0,\\qquad
\\frac{\\partial(hu)}{\\partial t}+\\frac{\\partial}{\\partial x}\\!\\left(hu^2+\\frac{g h^2}{2}\\right)
= gh(S_0-S_f)$$
where $S_0$ = bed slope, $S_f=n^2u^2/h^{4/3}$ = friction slope (Manning).  

**Manning's equation:** $Q = \\frac{1}{n}AR^{2/3}S^{1/2}$,  
$R=A/P$ = hydraulic radius, $A$ = cross-section area, $P$ = wetted perimeter.

In [ ]:
# ── SymPy: Manning's equation ─────────────────────────────────────
Q_s,n_s,A_s,R_s,S_s = sp.symbols('Q n A R S', positive=True)
h_s,b_s,z_s         = sp.symbols('h b z', positive=True)  # depth, base width, side slope

manning_eq = sp.Eq(Q_s, (1/n_s)*A_s*R_s**sp.Rational(2,3)*sp.sqrt(S_s))
print('Manning equation:'); display(Math(sp.latex(manning_eq)))

# Trapezoidal channel: A=bh+zh², P=b+2h√(1+z²), R=A/P
A_trap = b_s*h_s + z_s*h_s**2
P_trap = b_s + 2*h_s*sp.sqrt(1+z_s**2)
R_trap = sp.simplify(A_trap/P_trap)
print('\nTrapezoidal channel (b=base, z=side slope):')
display(Math(r'A = '+sp.latex(A_trap)+r',\quad P = '+sp.latex(P_trap)))
display(Math(r'R = A/P = '+sp.latex(R_trap)))

# Optimal (most efficient) section: dP/dh = 0 at fixed A
h_opt_eq = sp.solve(sp.diff(P_trap,h_s),h_s)
print('\nOptimal depth for minimum P (most efficient section):')
for sol in h_opt_eq: display(Math(r'h_{opt} = '+sp.latex(sp.simplify(sol))))

# ── Numerical: rating curves for channel types ─────────────────────
n_man  = 0.035   # natural stream
S0_val = 0.002   # 0.2% bed slope
h_vals = np.linspace(0.01, 3.0, 300)  # depth 0–3 m

channels = [
    ('Rectangular b=5m',     lambda h: (5*h,          5+2*h)),
    ('Trapezoidal b=3m z=1', lambda h: (3*h+h**2,     3+2*h*np.sqrt(2))),
    ('Trapezoidal b=3m z=2', lambda h: (3*h+2*h**2,   3+2*h*np.sqrt(5))),
    ('Circular D=4m',        lambda h: (8*(np.arccos(1-h/2)-
                                          (1-h/2)*np.sqrt(h*(4-h)))/1,
                                         4*np.arccos(1-h/2))),
]

fig,axes=plt.subplots(1,3,figsize=(14,4))
for name,geom_fn in channels:          # loop: channel types
    h_lim = h_vals if 'Circ' not in name else h_vals[h_vals<=4.0]
    A_arr,P_arr = geom_fn(h_lim)
    R_arr = A_arr/(P_arr+1e-10)
    Q_arr = (1/n_man)*A_arr*R_arr**(2/3)*np.sqrt(S0_val)
    V_arr = Q_arr/(A_arr+1e-10)
    Fr_arr= V_arr/np.sqrt(9.81*h_lim)  # Froude number
    axes[0].plot(Q_arr,h_lim,lw=2,label=name)
    axes[1].plot(V_arr,h_lim,lw=2,label=name)
    axes[2].plot(Fr_arr,h_lim,lw=2,label=name)

for ax,xl,tit in [(axes[0],'Q (m³/s)','Rating curve Q(h)'),
                   (axes[1],'V (m/s)','Velocity V(h)'),
                   (axes[2],'Fr','Froude number Fr(h)')]:
    ax.set_xlabel(xl); ax.set_ylabel('Depth h (m)')
    ax.set_title(tit); ax.legend(fontsize=7); ax.grid(True,alpha=0.3)
axes[2].axvline(1,ls='--',color='k',lw=1.5,label='Fr=1 (critical)')
plt.suptitle('🌊 Manning Rating Curves',fontsize=12,fontweight='bold')
plt.tight_layout(); plt.show()

# ── 1D shallow water: dam-break (Riemann problem) ─────────────────
# FTCS on Lax-Wendroff scheme
Nx   = 400; Lx=200.0; dx=Lx/Nx; dt=0.02; g_sw=9.81
x_sw = np.linspace(0,Lx,Nx)

# Initial condition: dam at x=100m
h_sw = np.where(x_sw<100, 3.0, 0.5)   # left deeper
u_sw = np.zeros(Nx)

def lax_wendroff_step(h, u, dx, dt, g, n_man_sw=0.03, S0=0.001):
    """One time step of Lax-Wendroff scheme for 1D SWE."""
    hu = h*u
    # Fluxes F = (hu, hu²+gh²/2)
    F1 = hu
    F2 = hu**2/(h+1e-10) + 0.5*g*h**2
    # Lax-Wendroff: predictor (half step at cell edges)
    h_e  = 0.5*(h[1:]+h[:-1]) - 0.5*dt/dx*(F1[1:]-F1[:-1])
    hu_e = 0.5*(hu[1:]+hu[:-1]) - 0.5*dt/dx*(F2[1:]-F2[:-1])
    u_e  = hu_e/(h_e+1e-10)
    # Corrector (full step)
    F1e = hu_e; F2e = hu_e**2/(h_e+1e-10)+0.5*g*h_e**2
    h_new  = np.zeros(Nx); hu_new = np.zeros(Nx)
    h_new[1:-1]  = h[1:-1]  - dt/dx*(F1e[1:]-F1e[:-1])
    hu_new[1:-1] = hu[1:-1] - dt/dx*(F2e[1:]-F2e[:-1])
    # Friction source term (Manning)
    Sf = n_man_sw**2 * u[1:-1]*np.abs(u[1:-1]) / (h[1:-1]**(4/3)+1e-10)
    hu_new[1:-1] += dt*g*h[1:-1]*(S0 - Sf)
    # Reflective BCs
    h_new[0]=h_new[1]; h_new[-1]=h_new[-2]
    hu_new[0]=0; hu_new[-1]=0
    u_new = hu_new/(h_new+1e-10)
    return h_new.clip(1e-4), u_new

snapshots_sw=[]
T_total=20.0; n_steps=int(T_total/dt)
h_cur,u_cur=h_sw.copy(),u_sw.copy()
save_times={0,int(n_steps*0.1),int(n_steps*0.25),int(n_steps*0.5),int(n_steps*0.75),n_steps-1}

for step_i in range(n_steps):          # loop: SWE time integration
    h_cur,u_cur = lax_wendroff_step(h_cur,u_cur,dx,dt,g_sw)
    if step_i in save_times: snapshots_sw.append((step_i*dt,h_cur.copy(),u_cur.copy()))

fig,axes=plt.subplots(1,2,figsize=(13,4))
colors_sw=plt.cm.Blues(np.linspace(0.3,1.0,len(snapshots_sw)))
for (t_v,h_v,u_v),col in zip(snapshots_sw,colors_sw):   # loop: plot snapshots
    axes[0].plot(x_sw,h_v,color=col,lw=1.8,label=f't={t_v:.1f}s')
    axes[1].plot(x_sw,u_v,color=col,lw=1.8)
axes[0].set_xlabel('x (m)'); axes[0].set_ylabel('Depth h (m)')
axes[0].set_title('Dam-break: water depth'); axes[0].legend(fontsize=8)
axes[0].grid(True,alpha=0.3)
axes[1].set_xlabel('x (m)'); axes[1].set_ylabel('Velocity u (m/s)')
axes[1].set_title('Dam-break: flow velocity'); axes[1].grid(True,alpha=0.3)
plt.suptitle('🌊 1D Shallow Water: Dam-Break (Lax-Wendroff)',fontsize=12,fontweight='bold')
plt.tight_layout(); plt.show()

---
# §3 🏗️ Concrete — Stress-Strain, Compressive Strength, RC Beam Design

**Concrete compressive strength** $f'_c$ controls most design.  
**Hognestad parabola:** $f_c = f'_c\\left[\\frac{2\\varepsilon}{\\varepsilon_0}-\\left(\\frac{\\varepsilon}{\\varepsilon_0}\\right)^2\\right]$,  
$\\varepsilon_0=2f'_c/E_c$, $E_c=57000\\sqrt{f'_c}$ psi (ACI 318).  
**RC beam (ACI):** $M_n = A_s f_y\\left(d-\\frac{a}{2}\\right)$, $a=A_sf_y/(0.85f'_cb)$.

In [ ]:
# ── SymPy: Hognestad parabola + RC beam ──────────────────────────
eps_c, eps_0 = sp.symbols('varepsilon varepsilon_0', positive=True)
fc_s, fcp_s  = sp.symbols('f_c f_c_prime', positive=True)
E_c_s        = sp.Symbol('E_c', positive=True)

hognestad = sp.Eq(fc_s, fcp_s*(2*eps_c/eps_0 - (eps_c/eps_0)**2))
print('Hognestad parabola:'); display(Math(sp.latex(hognestad)))

# Peak strain
d_fc = sp.diff(hognestad.rhs, eps_c)
eps_peak = sp.solve(d_fc, eps_c)[0]
print('Peak at ε₀:'); display(Math(r'\varepsilon_{peak} = '+sp.latex(eps_peak)))

# ACI E_c (metric: MPa, mm)
fcp_MPa, w_c = sp.symbols('f_c_prime w_c', positive=True)
E_c_ACI = 0.043 * w_c**1.5 * sp.sqrt(fcp_MPa)   # MPa, w_c in kg/m³
print('\nACI 318 E_c (MPa, w_c in kg/m³):')
display(Math(r'E_c = 0.043\,w_c^{1.5}\sqrt{f\'_c} = '+sp.latex(E_c_ACI)))

# RC beam moment capacity
As_s,fy_s,d_s,b_s = sp.symbols('A_s f_y d b', positive=True)
a_depth = As_s*fy_s/(0.85*fcp_s*b_s)
Mn = As_s*fy_s*(d_s - a_depth/2)
print('\nRC beam nominal moment (ACI):')
display(Math(r'a = '+sp.latex(a_depth)))
display(Math(r'M_n = A_sf_y(d-a/2) = '+sp.latex(sp.expand(Mn))))

# ── Numerical: stress-strain curves for different f'c ─────────────
fcp_vals = np.array([20,30,40,50,60,80])  # MPa
w_c_val  = 2400.0  # kg/m³ (normal weight concrete)

fig,axes=plt.subplots(1,3,figsize=(14,4))

for fcp_v in fcp_vals:             # loop: concrete grades
    Ec_v   = 0.043*w_c_val**1.5*np.sqrt(fcp_v)  # MPa
    eps0_v = 2*fcp_v/Ec_v
    eps_arr= np.linspace(0, 0.0035, 500)  # up to ultimate 3.5‰
    # Hognestad ascending + linear descending
    fc_arr = np.where(eps_arr<=eps0_v,
                      fcp_v*(2*eps_arr/eps0_v-(eps_arr/eps0_v)**2),
                      fcp_v*(1-0.15*(eps_arr-eps0_v)/(0.0035-eps0_v)))
    fc_arr = fc_arr.clip(0)
    axes[0].plot(eps_arr*1000, fc_arr, lw=1.8, label=f'f\'c={fcp_v}')

axes[0].set_xlabel('Strain ε (‰)'); axes[0].set_ylabel('Stress f_c (MPa)')
axes[0].set_title('Hognestad + descending branch'); axes[0].legend(fontsize=7)
axes[0].grid(True,alpha=0.3)

# RC beam: moment-curvature (M-κ) for b=300mm, d=500mm, As varying
b_mm=300; d_mm=500; fy_v=420  # MPa
As_vals_mm2 = np.array([500,800,1200,1600,2200])  # mm²
fcp_beam = 30.0  # MPa
Ec_beam  = 0.043*2400**1.5*np.sqrt(fcp_beam)
Es_v     = 200000.0  # MPa steel

print('\nRC Beam Design Table (b=300mm, d=500mm, f\'c=30MPa, fy=420MPa):')
print(f'{"As(mm²)":10}  {"a(mm)":8}  {"Mn(kN·m)":10}  {"φMn(kN·m)":10}  {"ρ(%)"}:')
for As_v in As_vals_mm2:           # loop: beam designs
    a_v  = As_v*fy_v/(0.85*fcp_beam*b_mm)
    Mn_v = As_v*fy_v*(d_mm-a_v/2)*1e-6  # kN·m
    phi  = 0.90
    rho  = 100*As_v/(b_mm*d_mm)
    axes[1].bar(As_v, phi*Mn_v, width=80, alpha=0.7,
                color=plt.cm.RdYlGn(rho/2))
    print(f'{As_v:10.0f}  {a_v:8.1f}  {Mn_v:10.1f}  {phi*Mn_v:10.1f}  {rho:.2f}')
axes[1].set_xlabel('As (mm²)'); axes[1].set_ylabel('φMn (kN·m)')
axes[1].set_title('RC Beam capacity vs steel area')
axes[1].grid(True,alpha=0.3,axis='y')

# Creep (ACI 209): J(t) = (1/Ec)(1 + φ_c(t))
# φ_c(t) = t^0.6/(10+t^0.6) * φ_u
phi_u = 2.35  # ultimate creep coefficient
t_yrs = np.linspace(0, 50, 500)
t_days= t_yrs*365
phi_t = t_days**0.6/(10+t_days**0.6) * phi_u
Ec_gpa= Ec_beam/1000
J_t   = (1 + phi_t)/Ec_gpa   # compliance GPa⁻¹

axes[2].plot(t_yrs, phi_t, 'b-', lw=2, label='φ(t) creep coeff')
ax2_twin = axes[2].twinx()
ax2_twin.plot(t_yrs, J_t, 'r--', lw=2, label='J(t) compliance')
axes[2].set_xlabel('Time (years)'); axes[2].set_ylabel('φ(t)', color='b')
ax2_twin.set_ylabel('J(t) (GPa⁻¹)', color='r')
axes[2].set_title('Concrete creep (ACI 209)')
axes[2].grid(True,alpha=0.3)
lines1,lbs1=axes[2].get_legend_handles_labels()
lines2,lbs2=ax2_twin.get_legend_handles_labels()
axes[2].legend(lines1+lines2, lbs1+lbs2, fontsize=8)
plt.suptitle('🏗️ Concrete: Stress-Strain · RC Beam · Creep',fontsize=12,fontweight='bold')
plt.tight_layout(); plt.show()

---
# §4 🪵 Wood — Orthotropic Elasticity, Grain Texture, Weibull Failure

Wood is **orthotropic**: 3 principal directions — **L**ongitudinal (grain),  
**R**adial, **T**angential. 9 independent elastic constants.  

$$\\begin{pmatrix}\\varepsilon_L\\\\\\varepsilon_R\\\\\\varepsilon_T\\end{pmatrix} =
\\begin{pmatrix}1/E_L & -\\nu_{RL}/E_R & -\\nu_{TL}/E_T \\\\
-\\nu_{LR}/E_L & 1/E_R & -\\nu_{TR}/E_T \\\\
-\\nu_{LT}/E_L & -\\nu_{RT}/E_R & 1/E_T\\end{pmatrix}
\\begin{pmatrix}\\sigma_L\\\\\\sigma_R\\\\\\sigma_T\\end{pmatrix}$$

**Weibull modulus $m$:** brittle failure probability $P_f=1-\\exp[-(\\sigma/\\sigma_0)^m]$.

In [ ]:
# ── Orthotropic compliance matrix for Douglas Fir ─────────────────
# Values from Forest Products Lab (GPa, dimensionless)
wood_species = {
    'Douglas Fir':  dict(EL=13.7, ER=0.91, ET=0.50,
                         GLR=0.88, GLT=0.74, GRT=0.095,
                         nuLR=0.292, nuLT=0.449, nuRT=0.390),
    'Southern Pine':dict(EL=14.7, ER=1.46, ET=0.91,
                         GLR=1.15, GLT=0.74, GRT=0.18,
                         nuLR=0.328, nuLT=0.292, nuRT=0.382),
    'Balsa':        dict(EL=3.70, ER=0.23, ET=0.15,
                         GLR=0.14, GLT=0.10, GRT=0.05,
                         nuLR=0.230, nuLT=0.490, nuRT=0.600),
}

def compliance_matrix(p):
    """6×6 compliance S matrix (GPa⁻¹). Order: L,R,T,LR,LT,RT."""
    S = np.zeros((6,6))
    # Normal compliance
    S[0,0]=1/p['EL']; S[1,1]=1/p['ER']; S[2,2]=1/p['ET']
    # Poisson coupling (symmetric)
    S[0,1]=S[1,0]=-p['nuLR']/p['EL']
    S[0,2]=S[2,0]=-p['nuLT']/p['EL']
    S[1,2]=S[2,1]=-p['nuRT']/p['ER']
    # Shear
    S[3,3]=1/p['GLR']; S[4,4]=1/p['GLT']; S[5,5]=1/p['GRT']
    return S

print('Compliance matrices S (GPa⁻¹):')
for sp_name, p in wood_species.items():    # loop: species
    S = compliance_matrix(p)
    C = np.linalg.inv(S[:3,:3])  # 3×3 stiffness (normal only)
    print(f'\n{sp_name}:')
    print(f'  E_L/E_T anisotropy ratio: {p["EL"]/p["ET"]:.1f}')
    print(f'  S matrix (normal 3×3, GPa⁻¹):\n{S[:3,:3].round(4)}')

# ── Wood grain texture (procedural) ─────────────────────────────────
def wood_grain_texture(H,W,seed=7,rings=25,waviness=0.08):
    """Procedural wood grain: annual rings + ray fleck + knots."""
    rng_w = np.random.default_rng(seed)
    x,y   = np.meshgrid(np.linspace(-1,1,W),np.linspace(-1,1,H))
    # Add turbulence for waviness
    noise = perlin2d_oct((H,W),octaves=4,persistence=0.6,seed=seed+10)
    x_w   = x + waviness*noise
    y_w   = y + waviness*gaussian_filter(noise,sigma=3)
    # Annual rings: concentric ellipses with center offset
    cx,cy = rng_w.uniform(-0.3,0.3,2)
    r_ring= np.sqrt((x_w-cx)**2*1.4 + (y_w-cy)**2)
    ring_val = 0.5+0.5*np.sin(rings*np.pi*r_ring)
    # Ray fleck: thin radial lines
    angle_val = np.arctan2(y_w-cy, x_w-cx)/(2*np.pi)
    fleck = 0.1*(1+np.sin(120*angle_val))
    # Knot: one circular dark spot
    kx,ky = rng_w.uniform(-0.5,0.5,2)
    knot  = np.exp(-((x-kx)**2+(y-ky)**2)/0.003)
    grain = (ring_val*(1+0.4*noise) + fleck).clip(0,1)
    grain = (grain - knot*0.5).clip(0,1)
    return grain

grain_df  = wood_grain_texture(256,512,seed=42)
grain_oak = wood_grain_texture(256,512,seed=13,rings=35,waviness=0.12)

# ── Weibull failure statistics ─────────────────────────────────────
m_weibull  = 8.0    # Weibull modulus for Douglas Fir MOR
sigma0_val = 50.0   # scale parameter (MPa)
sigma_arr  = np.linspace(0, 100, 400)
Pf_arr     = 1 - np.exp(-(sigma_arr/sigma0_val)**m_weibull)

# Weibull for different moisture contents
mc_vals = [8, 12, 15, 19]  # % moisture content
# Strength drops ~1% per 1% MC above ~9% (empirical)

fig,axes=plt.subplots(1,3,figsize=(14,4))

axes[0].imshow(grain_df, cmap='YlOrBr', origin='lower', aspect='auto')
axes[0].set_title('Procedural Douglas Fir grain (256×512)')
axes[0].axis('off')

axes[1].imshow(grain_oak, cmap='RdYlBr', origin='lower', aspect='auto', vmin=0.1)
axes[1].set_title('Procedural Oak grain (high waviness)')
axes[1].axis('off')

for mc_v in mc_vals:               # loop: moisture content
    # Adjust sigma0 for moisture
    s0_adj = sigma0_val * (1 - 0.012*max(mc_v-9,0))
    Pf_mc  = 1-np.exp(-(sigma_arr/s0_adj)**m_weibull)
    axes[2].plot(sigma_arr, Pf_mc*100, lw=2, label=f'MC={mc_v}%')

axes[2].axhline(50,ls='--',color='k',lw=1,label='P_f=50%')
axes[2].set_xlabel('Bending stress σ (MPa)')
axes[2].set_ylabel('Failure probability (%)')
axes[2].set_title(f'Weibull MOR (m={m_weibull}, σ₀={sigma0_val}MPa)')
axes[2].legend(fontsize=8); axes[2].grid(True,alpha=0.3)
plt.suptitle('🪵 Wood: Orthotropic Elasticity + Grain Texture + Weibull',
             fontsize=12,fontweight='bold')
plt.tight_layout(); plt.show()

# Print stiffness comparison
print('\nWood species elastic moduli (GPa):')
print(f'{"Species":16}  {"EL":6}  {"ER":6}  {"ET":6}  {"EL/ET":6}')
for sp_name,p in wood_species.items():  # loop: print table
    print(f'{sp_name:16}  {p["EL"]:6.2f}  {p["ER"]:6.2f}  {p["ET"]:6.2f}  {p["EL"]/p["ET"]:6.1f}')

---
# §5 🏘️ Pervious Neighborhood — SCS Curve Number, Green-Ampt, Stormwater

**SCS Curve Number** method: $Q = (P-I_a)^2/(P-I_a+S)$, $S=25400/CN-254$ (mm),
$I_a=0.2S$.  
**Green-Ampt infiltration:** $f=K_s(1+\\psi\\Delta\\theta/F)$, $F=$ cumulative infiltration.  
**Pervious pavement** reduces CN by 20–30 points → dramatically less runoff.

In [ ]:
# ── SCS Curve Number runoff ────────────────────────────────────────
# Land use CN values (AMC-II, good condition)
land_use_CN = {
    'Impervious road/roof':   98,
    'Conventional pavement':  98,
    'Pervious pavement':      70,
    'Lawn (good)':            61,
    'Forest (good)':          36,
    'Meadow':                 30,
    'Open water':             99,
}

P_vals_mm = np.linspace(0, 150, 400)   # rainfall 0-150mm

def scs_runoff(P_mm, CN):
    S = 25400/CN - 254   # mm
    Ia = 0.2*S           # initial abstraction
    Q  = np.where(P_mm > Ia,
                  (P_mm-Ia)**2/(P_mm-Ia+S), 0.0)
    return Q

# Neighborhood scenario: mix of land uses
neighborhood_A = dict(  # conventional
    imp_road=0.15, conv_pave=0.35, lawn=0.30, forest=0.10, open=0.10)
neighborhood_B = dict(  # pervious redesign
    imp_road=0.15, perv_pave=0.35, lawn=0.20, forest=0.20, open=0.10)

lu_map_A = {'imp_road':98,'conv_pave':98,'lawn':61,'forest':36,'open':99}
lu_map_B = {'imp_road':98,'perv_pave':70,'lawn':61,'forest':36,'open':99}

CN_A = sum(f*lu_map_A[k] for k,f in neighborhood_A.items())
CN_B = sum(f*lu_map_B[k] for k,f in neighborhood_B.items())
Q_A  = scs_runoff(P_vals_mm, CN_A)
Q_B  = scs_runoff(P_vals_mm, CN_B)
print(f'Conventional neighborhood: CN={CN_A:.1f}')
print(f'Pervious neighborhood:     CN={CN_B:.1f}')
print(f'Runoff reduction at 50mm rain: {100*(Q_A[200]-Q_B[200])/(Q_A[200]+1e-10):.1f}%')

# ── Green-Ampt infiltration ────────────────────────────────────────
# Parameters for different soils
soils = {
    'Sandy loam':    dict(Ks=1.09e-5, psi=113e-3, dtheta=0.412-0.105),   # m/s, m
    'Loam':          dict(Ks=3.67e-6, psi=88.9e-3,dtheta=0.464-0.116),
    'Clay loam':     dict(Ks=1.17e-6, psi=208e-3, dtheta=0.471-0.155),
    'Clay':          dict(Ks=3.3e-7,  psi=316e-3, dtheta=0.479-0.201),
    'Pervious pave': dict(Ks=5.0e-5,  psi=50e-3,  dtheta=0.30),
}

def green_ampt(t_arr, Ks, psi, dtheta, i_rain=3e-5):
    """Green-Ampt infiltration via implicit Newton iteration."""
    dt_ga = t_arr[1]-t_arr[0]
    F = 1e-6; f_rate=[]; F_arr=[]
    for t_v in t_arr:               # loop: time steps
        # Implicit: F_new = Ks*dt + psi*dtheta*ln(1+F_new/(psi*dtheta)) + F_old
        for _ in range(20):         # Newton inner loop
            lhs    = F
            rhs    = F - dt_ga*Ks - F + Ks*dt_ga + psi*dtheta*np.log(1+F/(psi*dtheta+1e-15))+\
                     (F_arr[-1] if F_arr else 1e-6)
            dF     = Ks*(1 + psi*dtheta/(F+1e-10))
            F_new  = F + (Ks*dt_ga + psi*dtheta*np.log(1+F/(psi*dtheta+1e-15)) -
                          (F-(F_arr[-1] if F_arr else 0)))/max(1-1,1)
            break
        f_rate_v = Ks*(1 + psi*dtheta/(F+1e-10))
        actual_inf = min(f_rate_v, i_rain)  # can't infiltrate more than rainfall
        F += actual_inf*dt_ga
        f_rate.append(f_rate_v); F_arr.append(F)
    return np.array(f_rate), np.array(F_arr)

t_ga = np.linspace(0, 3600, 360)   # 1 hour, 10s steps
i_rain_val = 3e-5  # 30 mm/hr = 3e-5 m/s

fig,axes=plt.subplots(1,3,figsize=(14,4))

# SCS runoff curves
for lu_name,(cn_v) in land_use_CN.items():  # loop: land uses
    Q_lu = scs_runoff(P_vals_mm, cn_v)
    axes[0].plot(P_vals_mm,Q_lu,lw=1.5,label=f'{lu_name} (CN={cn_v})')
axes[0].set_xlabel('Rainfall P (mm)'); axes[0].set_ylabel('Runoff Q (mm)')
axes[0].set_title('SCS Curve Number runoff'); axes[0].legend(fontsize=6)
axes[0].grid(True,alpha=0.3)

# Neighborhood comparison
axes[1].fill_between(P_vals_mm,Q_A,Q_B,alpha=0.3,color='red',label='Runoff reduction')
axes[1].plot(P_vals_mm,Q_A,'r-',lw=2,label=f'Conventional CN={CN_A:.1f}')
axes[1].plot(P_vals_mm,Q_B,'g-',lw=2,label=f'Pervious CN={CN_B:.1f}')
axes[1].set_xlabel('Rainfall P (mm)'); axes[1].set_ylabel('Runoff Q (mm)')
axes[1].set_title('Pervious vs conventional neighborhood')
axes[1].legend(fontsize=8); axes[1].grid(True,alpha=0.3)

# Green-Ampt infiltration rates
for soil_name,s in soils.items():   # loop: soil types
    f_r,F_c = green_ampt(t_ga,s['Ks'],s['psi'],s['dtheta'],i_rain_val)
    axes[2].plot(t_ga/60, f_r*1000*3600, lw=2, label=soil_name)   # mm/hr
axes[2].axhline(i_rain_val*1000*3600,ls='--',color='k',lw=1.5,
                 label=f'Rainfall i={i_rain_val*3.6e6:.0f}mm/hr')
axes[2].set_xlabel('Time (min)'); axes[2].set_ylabel('Infiltration rate (mm/hr)')
axes[2].set_title('Green-Ampt: f(t) for soil types')
axes[2].legend(fontsize=7); axes[2].grid(True,alpha=0.3)
plt.suptitle('🏘️ Pervious Neighborhood: SCS · Green-Ampt',fontsize=12,fontweight='bold')
plt.tight_layout(); plt.show()

---
# §6 💡 Light Pipe — TIR, Numerical Aperture, Luminance Transport, Étendue

A **light pipe** guides light by total internal reflection (TIR).  

**Critical angle:** $\\theta_c = \\arcsin(n_2/n_1)$.  
**Numerical aperture:** $\\text{NA} = n_0\\sin\\theta_{acc} = \\sqrt{n_1^2-n_2^2}$.  
**Étendue (invariant):** $E = n^2 A \\Omega = \\text{const}$ along lossless pipe.  
**Attenuation** (Fresnel losses at each bounce):
$L(z) = L_0 \\cdot R^{N(z)}$, $N(z) = z/(2a\\tan\\alpha)$.

In [ ]:
# ── SymPy: TIR and NA ─────────────────────────────────────────────
n1_s,n2_s,theta_s = sp.symbols('n_1 n_2 theta', positive=True)
NA_s = sp.Symbol('NA', positive=True)

# Snell's law at critical angle: n1 sin(θc) = n2 sin(90°) = n2
theta_c = sp.asin(n2_s/n1_s)
display(Math(r'\theta_c = \arcsin\!\left(\frac{n_2}{n_1}\right) = '+sp.latex(theta_c)))

# NA
NA_expr = sp.sqrt(n1_s**2 - n2_s**2)
display(Math(r'\mathrm{NA} = \sqrt{n_1^2-n_2^2} = '+sp.latex(NA_expr)))

# Fresnel reflectance at TIR interface (s-polarization near critical)
# For angle of incidence θ_i (internal), rs = |(n1 cosθ - n2 cosθt)/(n1 cosθ + n2 cosθt)|²
theta_i_s = sp.Symbol('theta_i', positive=True)
cos_t     = sp.sqrt(1-(n1_s/n2_s*sp.sin(theta_i_s))**2)
rs = ((n1_s*sp.cos(theta_i_s)-n2_s*cos_t)/(n1_s*sp.cos(theta_i_s)+n2_s*cos_t))**2
print('\nFresnel R_s near critical angle:'); display(Math(r'R_s = '+sp.latex(rs)))

# ── Numerical: pipe parameters ─────────────────────────────────────
pipe_materials = {
    'PMMA in air':        dict(n1=1.492, n2=1.000, label='PMMA/air'),
    'Glass in air':       dict(n1=1.520, n2=1.000, label='Glass/air'),
    'PMMA/fluoropolymer': dict(n1=1.492, n2=1.400, label='PMMA/FEP'),
    'High-index glass':   dict(n1=1.700, n2=1.450, label='HiIdx/clad'),
    'Diamond/air':        dict(n1=2.417, n2=1.000, label='Diamond'),
}

print('\nLight Pipe Parameters:')
print(f'{"Material":24}  {"n1":5}  {"n2":5}  {"θc(°)":7}  {"NA":7}  {"θacc(°)":8}')
for name,m in pipe_materials.items():   # loop: materials
    tc = np.degrees(np.arcsin(m['n2']/m['n1']))
    NA = np.sqrt(m['n1']**2-m['n2']**2)
    ta = np.degrees(np.arcsin(NA))
    print(f'{name:24}  {m["n1"]:5.3f}  {m["n2"]:5.3f}  {tc:7.2f}  {NA:7.4f}  {ta:8.2f}')

# ── Ray tracing inside a rectangular light pipe ────────────────────
class LightPipe:
    def __init__(self, n1, n2, width, length, n_bounces_max=500):
        self.n1=n1; self.n2=n2
        self.w=width; self.L=length
        self.theta_c=np.arcsin(n2/n1)
        self.n_max=n_bounces_max

    def fresnel_R(self, theta_i):
        """Fresnel reflectance (average s+p) at internal interface."""
        if theta_i >= self.theta_c: return 1.0  # TIR
        cos_i = np.cos(theta_i)
        cos_t = np.sqrt(1-(self.n1/self.n2*np.sin(theta_i))**2+0j).real
        rs = abs((self.n1*cos_i-self.n2*cos_t)/(self.n1*cos_i+self.n2*cos_t))**2
        rp = abs((self.n2*cos_i-self.n1*cos_t)/(self.n2*cos_i+self.n1*cos_t))**2
        return (rs+rp)/2

    def trace_ray(self, y0, angle_rad):
        """Trace one ray. Returns (exit_z, final_intensity, path)."""
        y=y0; alpha=angle_rad  # angle to pipe axis
        intensity=1.0; z=0.0; path=[(z,y)]
        for bounce in range(self.n_max):  # loop: bounces
            # Distance to next wall
            if np.abs(np.sin(alpha))<1e-12: break  # paraxial → no bounce
            if np.sin(alpha)>0:
                dz=(self.w/2-y)/np.tan(alpha); y_hit=self.w/2
            else:
                dz=(-self.w/2-y)/np.tan(alpha); y_hit=-self.w/2
            z+=abs(dz)
            if z>=self.L: break
            R=self.fresnel_R(np.pi/2-abs(alpha))
            intensity*=R
            y=y_hit; alpha=-alpha  # reflect
            path.append((z,y))
        path.append((self.L,y+np.tan(alpha)*(self.L-z+abs(dz))-np.tan(alpha)*abs(dz)))
        return z,intensity,path

# PMMA pipe: 10mm wide, 300mm long
pipe = LightPipe(n1=1.492, n2=1.0, width=10e-3, length=300e-3)

# Trace fan of rays from center at various angles
angles_deg = np.linspace(-38,38,120)   # ±38° (just beyond critical 42°)
exit_intens=[]; exit_z=[]
selected_paths=[]

for ang_d in angles_deg:              # loop: ray fan
    ang_r = np.radians(ang_d)
    _,inten,path = pipe.trace_ray(0.0, ang_r)
    exit_intens.append(inten)
    if abs(ang_d) in [5,15,25,35]: selected_paths.append((ang_d,path))

# ── Étendue conservation ──────────────────────────────────────────
# E = A × Ω = constant. If pipe narrows: angle must grow.
A1_vals   = np.linspace(1,20,200)   # mm² input area
Omega1    = np.pi * np.sin(np.radians(20))**2   # input solid angle (sr)
A2_vals   = 5.0   # mm² output area (fixed)
# Required output angle: A1*Ω1 = A2*Ω2
Omega2    = A1_vals*Omega1/A2_vals
theta_out = np.degrees(np.arcsin(np.sqrt(Omega2/np.pi).clip(0,0.9999)))

fig,axes=plt.subplots(1,3,figsize=(15,4))

# Ray diagram
for ang_d,path in selected_paths:     # loop: draw selected rays
    zs_r=[p[0]*1000 for p in path]
    ys_r=[p[1]*1000 for p in path]
    col='b' if abs(ang_d)<pipe.theta_c*180/np.pi else 'r'
    axes[0].plot(zs_r,ys_r,color=col,lw=1.2,alpha=0.8,
                 label=f'{ang_d:.0f}°' if ang_d>0 else None)
# Pipe walls
axes[0].axhline(pipe.w/2*1000,color='k',lw=2)
axes[0].axhline(-pipe.w/2*1000,color='k',lw=2)
axes[0].set_xlabel('z (mm)'); axes[0].set_ylabel('y (mm)')
axes[0].set_title(f'PMMA light pipe (θc={np.degrees(pipe.theta_c):.1f}°)\nBlue=TIR, Red=leaky')
axes[0].set_xlim(0,pipe.L*1000); axes[0].grid(True,alpha=0.2)

# Exit intensity vs angle
axes[1].plot(angles_deg,np.array(exit_intens)*100,'b-',lw=2)
axes[1].axvline(np.degrees(pipe.theta_c),ls='--',color='r',lw=1.5,
                 label=f'θc={np.degrees(pipe.theta_c):.1f}°')
axes[1].axvline(-np.degrees(pipe.theta_c),ls='--',color='r',lw=1.5)
axes[1].set_xlabel('Launch angle α (°)')
axes[1].set_ylabel('Exit intensity (%)')
axes[1].set_title('Light pipe efficiency vs launch angle')
axes[1].legend(fontsize=9); axes[1].grid(True,alpha=0.3)

# Étendue: output angle vs input area
axes[2].plot(A1_vals,theta_out,'g-',lw=2)
axes[2].axhline(90,ls='--',color='k',lw=1,alpha=0.5,label='90° limit')
axes[2].fill_between(A1_vals,theta_out,90,alpha=0.1,color='red',label='Unphysical')
axes[2].set_xlabel('Input area A₁ (mm²)')
axes[2].set_ylabel('Required output angle θ₂ (°)')
axes[2].set_title(f'Étendue: A₂={A2_vals}mm², θ₁=20°')
axes[2].legend(fontsize=8); axes[2].grid(True,alpha=0.3)
plt.suptitle('💡 Light Pipe: TIR · NA · Ray Tracing · Étendue',fontsize=12,fontweight='bold')
plt.tight_layout(); plt.show()

print(f'\nLight pipe summary (PMMA, n={pipe.n1}, {pipe.L*1000:.0f}mm long, {pipe.w*1000:.0f}mm wide):')
print(f'  Critical angle θc = {np.degrees(pipe.theta_c):.2f}°')
print(f'  NA = {np.sqrt(pipe.n1**2-pipe.n2**2):.4f}')
print(f'  Acceptance half-angle = {np.degrees(np.arcsin(np.sqrt(pipe.n1**2-pipe.n2**2))):.2f}°')